# LightGBM Model — S6E4 Irrigation Need
Mirrors the XGBoost notebook structure. Explores 3 hyperparameter sets, then trains a best model and generates a Kaggle submission.

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import balanced_accuracy_score, classification_report
from sklearn.utils.class_weight import compute_sample_weight
from lightgbm import LGBMClassifier

# Load data
train_df = pd.read_csv('train.csv')
test_df  = pd.read_csv('test.csv')

X = train_df.drop(columns=['id', 'Irrigation_Need'])
y_raw = train_df['Irrigation_Need']

# Encode target
target_encoder = LabelEncoder()
y = target_encoder.fit_transform(y_raw)
print('Classes:', target_encoder.classes_)

# LightGBM supports native categoricals when dtype is 'category'
cat_cols = X.select_dtypes(include=['object']).columns.tolist()
for col in cat_cols:
    X[col] = X[col].astype('category')

# Train / validation split (stratified, same seed as XGBoost notebook)
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# Sample weights to handle class imbalance (mirrors XGBoost approach)
sample_weights = compute_sample_weight(class_weight='balanced', y=y_train)

print(f'Training set: {X_train.shape}, Validation set: {X_val.shape}')

Classes: ['High' 'Low' 'Medium']


C:\Users\sydne\AppData\Local\Temp\ipykernel_5960\670522335.py:21: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = X.select_dtypes(include=['object']).columns.tolist()


Training set: (504000, 19), Validation set: (126000, 19)


## Hyperparameter Exploration

Three deliberately different configurations:

| Set | Key changes | Expected effect |
|-----|-------------|----------------|
| 1 | Few leaves (31), high LR, no regularisation | Fast, may underfit |
| 2 | More leaves (127), lower LR, L1+L2 regularisation | Better generalisation |
| 3 | Many leaves (255), very low LR, high min_child_samples, colsample | Slow, strong regularisation |

In [2]:
lgbm_param_sets = [
    # Set 1: shallow & fast — few leaves, high learning rate, no regularisation
    {
        'n_estimators': 100,
        'learning_rate': 0.15,
        'num_leaves': 31,
        'max_depth': -1,
        'reg_alpha': 0.0,
        'reg_lambda': 0.0,
        'class_weight': 'balanced',
    },
    # Set 2: medium complexity — more leaves, moderate LR, L1+L2 regularisation
    {
        'n_estimators': 300,
        'learning_rate': 0.05,
        'num_leaves': 127,
        'max_depth': -1,
        'reg_alpha': 1.0,
        'reg_lambda': 1.0,
        'class_weight': 'balanced',
    },
    # Set 3: complex & regularised — many leaves, very low LR, high min_child_samples, feature subsampling
    {
        'n_estimators': 500,
        'learning_rate': 0.02,
        'num_leaves': 255,
        'max_depth': -1,
        'reg_alpha': 3.0,
        'reg_lambda': 3.0,
        'min_child_samples': 50,
        'colsample_bytree': 0.8,
        'subsample': 0.8,
        'subsample_freq': 1,
        'class_weight': 'balanced',
    },
]

print('=== LightGBM Hyperparameter Exploration ===')
lgbm_results = []
for idx, params in enumerate(lgbm_param_sets, start=1):
    model = LGBMClassifier(random_state=42, verbose=-1, **params)
    model.fit(X_train, y_train, sample_weight=sample_weights)
    preds = model.predict(X_val)
    score = balanced_accuracy_score(y_val, preds)
    lgbm_results.append((idx, score, params, model))
    print(f'Set {idx}: balanced_accuracy = {score:.4f} | n_est={params["n_estimators"]}, '
          f'lr={params["learning_rate"]}, leaves={params["num_leaves"]}, '
          f'reg_alpha={params["reg_alpha"]}, reg_lambda={params["reg_lambda"]}')

best_lgbm_idx, best_lgbm_score, best_lgbm_params, best_lgbm_model = max(lgbm_results, key=lambda x: x[1])
print(f'\nBest set: {best_lgbm_idx} with balanced_accuracy = {best_lgbm_score:.4f}')

=== LightGBM Hyperparameter Exploration ===
Set 1: balanced_accuracy = 0.9656 | n_est=100, lr=0.15, leaves=31, reg_alpha=0.0, reg_lambda=0.0
Set 2: balanced_accuracy = 0.9715 | n_est=300, lr=0.05, leaves=127, reg_alpha=1.0, reg_lambda=1.0
Set 3: balanced_accuracy = 0.9708 | n_est=500, lr=0.02, leaves=255, reg_alpha=3.0, reg_lambda=3.0

Best set: 2 with balanced_accuracy = 0.9715


In [3]:
# Detailed classification report for the best set
print('--- Classification Report (Best LightGBM Set) ---')
best_preds = best_lgbm_model.predict(X_val)
print(classification_report(y_val, best_preds, target_names=target_encoder.classes_))

--- Classification Report (Best LightGBM Set) ---
              precision    recall  f1-score   support

        High       0.88      0.95      0.92      4202
         Low       0.99      0.99      0.99     73983
      Medium       0.99      0.97      0.98     47815

    accuracy                           0.98    126000
   macro avg       0.95      0.97      0.96    126000
weighted avg       0.98      0.98      0.98    126000



## Final Model — Retrain on Full Training Data & Generate Submission

In [4]:
print('--- Training Final LightGBM Model on Full Dataset ---')

# Reapply category dtype to full X (already done above, but explicit for clarity)
X_full = train_df.drop(columns=['id', 'Irrigation_Need'])
for col in cat_cols:
    X_full[col] = X_full[col].astype('category')

sample_weights_full = compute_sample_weight(class_weight='balanced', y=y)

final_lgbm_model = LGBMClassifier(random_state=42, verbose=-1, **best_lgbm_params)
final_lgbm_model.fit(X_full, y, sample_weight=sample_weights_full)

# Prepare test set — apply same category encoding
X_test = test_df.drop(columns=['id'])
for col in cat_cols:
    X_test[col] = X_test[col].astype('category')

lgbm_preds_encoded = final_lgbm_model.predict(X_test)
lgbm_preds_text = target_encoder.inverse_transform(lgbm_preds_encoded)

submission_lgbm = pd.DataFrame({
    'id': test_df['id'],
    'Irrigation_Need': lgbm_preds_text
})
submission_lgbm.to_csv('S6E4_lgbm_submission.csv', index=False)
print('Success! LightGBM submission saved to S6E4_lgbm_submission.csv')
print(submission_lgbm['Irrigation_Need'].value_counts())

--- Training Final LightGBM Model on Full Dataset ---
Success! LightGBM submission saved to S6E4_lgbm_submission.csv
Irrigation_Need
Low       159410
Medium    100563
High       10027
Name: count, dtype: int64
